# MedLift-3D — Kaggle pipeline

End-to-end run on a Kaggle notebook. Built around three Kaggle facts:

* **12-hour session cap** — training uses `--max-hours 11`, checkpoints, and
  resumes automatically when you rerun the same cell in a new session.
* **No internet by default** — the projector is pure PyTorch, so nothing beyond
  Kaggle's preinstalled stack is needed. Logging is CSV on disk, not wandb.
* **`/kaggle/working` is the only writable path** — auto-detected.

Enable **GPU (T4 x2 or P100)** in Settings → Accelerator before running.

> Order matters: run the gates first. Four failure modes in this problem are
> silent — they produce plausible loss curves and wrong reconstructions.

## 0 · Setup

Point `REPO` at the code. Either add this repository as a Kaggle *dataset* /
*GitHub* source, or clone it if internet is enabled.

In [ ]:
import os, sys, subprocess, pathlib

def find_input(slug, owner=None, base=pathlib.Path('/kaggle/input')):
    """Locate an attached input by slug.

    Kaggle mounts a dataset at /kaggle/input/<slug> on some accounts/dataset
    types and at /kaggle/input/datasets/<owner>/<slug> on others -- the layout
    is not consistent, so guessing one and hardcoding it breaks silently the
    moment it's wrong. This tries both, then falls back to a search, and on
    failure prints what IS actually attached rather than a bare path error.
    """
    candidates = [base / slug]
    if owner:
        candidates.append(base / 'datasets' / owner / slug)
    candidates += sorted(base.glob(f'datasets/*/{slug}'))
    candidates += sorted(base.glob(f'*/{slug}'))
    for c in candidates:
        if c.exists():
            return c
    have = sorted(str(p.relative_to(base)) for p in base.glob('*/*') if p.is_dir())
    have += sorted(str(p.relative_to(base)) for p in base.glob('*') if p.is_dir())
    raise FileNotFoundError(
        f"no input attached for slug={slug!r} owner={owner!r}. "
        f"Currently under /kaggle/input:\n  " + "\n  ".join(sorted(set(have)) or ["(nothing attached)"]))

try:
    REPO = find_input('medlift3d')                # attached as a Kaggle Dataset
except FileNotFoundError:
    REPO = pathlib.Path('/kaggle/working/medlift3d')
    if not REPO.exists():
        # Needs internet enabled; otherwise attach the repo as a dataset.
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/foyeznaeem/medlift3d.git', str(REPO)], check=True)

# Kaggle input is read-only, so work from a writable copy.
WORK = pathlib.Path('/kaggle/working')
if str(REPO).startswith('/kaggle/input'):
    subprocess.run(['cp', '-r', str(REPO), str(WORK / 'medlift3d')], check=True)
    REPO = WORK / 'medlift3d'

sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

import torch
print('repo   :', REPO)
print('torch  :', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  gpu {i}: {p.name}  {p.total_memory/2**30:.1f} GiB')


## 1 · Gates — run these before spending any GPU time

| Gate | Asserts | Guards against |
|---|---|---|
| G1 | one canonical grid, affine never dropped | comparing volumes on different grids |
| G2 | `fp` differentiable, `bp` its exact adjoint | a projection loss with **no gradient** |
| G3 | water cylinder integral = `mu_water·2R` | HU/density confusion, scale errors |
| G4 | a *perfect* reconstruction scores perfectly | metrics that cannot detect anything |

If any gate fails, stop. Every one of these failures is silent and invalidates
every downstream number.

In [ ]:
!python scripts/run_gates.py

### 1b · Smoke test on synthetic phantoms — optional, but cheap

`make_phantoms.py` builds synthetic chest phantoms with nodules, so the whole
pipeline can be run end to end *before* committing to a multi-hour download.
Worth doing once: it exercises training, reconstruction, evaluation and all
three experiments on data that takes minutes to generate, and it sets `DATA`
so you can carry straight on through sections 3-7 on phantoms.

Skip it and go to section 2 if you already trust the pipeline and want real
data now. Section 2 re-points `DATA` at LIDC-IDRI either way.


In [ ]:
# 1b -- synthetic phantoms; skip if you are going straight to real data.
DATA = '/kaggle/working/data/phantom'

!python scripts/make_phantoms.py \
    --n-cases 60 \
    --shape 256 256 256 --spacing 1.5 1.5 1.5 \
    --views-a 16 32 --views-b 15 \
    --nodules 3 --device cuda \
    --out {DATA}


## 2 · Data — LIDC-IDRI

The dataset this project is built on, fetched automatically from
[TCIA](https://www.cancerimagingarchive.net/collection/lidc-idri/), the
collection's own host. LIDC-IDRI is the only public source with **voxel-level
nodule contours from four independent radiologists**, which is what
`evaluate.py`'s Dice and volume error, `mdvc.py` and `hallucination.py` all
need; centroid-only collections cannot produce a segmentation ground truth.

What the collection page lists, and what this notebook does with each:

| TCIA asset | Size | Used? |
|---|---|---|
| Images (DICOM, CT/DX/CR) | 133 GB, 1,010 subjects, 1,308 series | **Yes** — CT series only, downloaded in batches |
| Radiologist Annotations/Segmentations (XML) | 8.62 MB | **Yes** — this is the four-reader ground truth |
| Nodule Counts by Patient (XLSX) | 40 KB | No — `pylidc` derives counts from the XML itself |
| Patient Diagnoses (XLS) | 45 KB | No — this project measures volume, not malignancy |

The 133 GB figure is the whole collection; nothing here downloads that. `BATCH`
in 2-ii sets how many CT series to pull (~100-300 MB each), and 2-iii deletes
the raw DICOM once each case is resampled to the canonical grid, so the working
directory stays bounded no matter how many batches you run.

Needs **Settings → Internet: On**.


In [ ]:
# 2-i -- dependencies, and the 8.62 MB radiologist annotation XML.
#
# TCIA's REST API needs no API key for public collections like LIDC-IDRI.
# The annotation zip's direct URL is the one thing on this page I could not
# verify, so this tries TCIA first and falls back to the copy inside the
# attached Kaggle dataset if that 404s -- and says which one it used.
%pip install -q pylidc

import io
import pathlib
import xml.etree.ElementTree as ET
import zipfile

import requests

XML_DIR = pathlib.Path('/kaggle/working/lidc_xml')
XML_DIR.mkdir(parents=True, exist_ok=True)

TCIA_XML_URLS = [
    'https://wiki.cancerimagingarchive.net/download/attachments/1966254/LIDC-XML-only.zip',
    'https://www.cancerimagingarchive.net/wp-content/uploads/LIDC-XML-only.zip',
]

def fetch_annotation_xml():
    if any(XML_DIR.rglob('*.xml')):
        return 'already present'
    for url in TCIA_XML_URLS:
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                z.extractall(XML_DIR)
            return f'TCIA: {url}'
        except Exception as e:                                      # noqa: BLE001
            print(f'  {url} -> {type(e).__name__}: {e}')
    try:                        # fall back to the attached Kaggle mirror
        src = find_input('the-cancer-imaging-archive-lidcidri', owner='justinkirby')
        found = list(src.rglob('*.xml'))
        if found:
            return f'attached Kaggle dataset: {src} ({len(found)} files)'
    except FileNotFoundError:
        pass
    raise RuntimeError(
        'could not obtain the annotation XML from TCIA or from an attached '
        'dataset. Download "Radiologist Annotations/Segmentations" by hand '
        'from https://www.cancerimagingarchive.net/collection/lidc-idri/ , '
        'attach it as a Kaggle dataset, and re-run.')

source = fetch_annotation_xml()
xml_paths = sorted(XML_DIR.rglob('*.xml'))
if not xml_paths:               # came from the Kaggle fallback
    xml_paths = sorted(find_input('the-cancer-imaging-archive-lidcidri',
                                  owner='justinkirby').rglob('*.xml'))
print(f'annotation XML from {source}: {len(xml_paths)} files')

def index_xml_by_series(paths):
    """{SeriesInstanceUid -> xml path}, read from each file's own contents.
    Matching on content rather than filename, because TCIA's XML naming is not
    something to rely on. LIDC XML is namespaced, so match the local tag name."""
    index = {}
    for p in paths:
        try:
            root = ET.parse(p).getroot()
        except ET.ParseError:
            continue
        for el in root.iter():
            if el.tag.rsplit('}', 1)[-1] == 'SeriesInstanceUid' and el.text:
                index[el.text.strip()] = p
                break
    return index

xml_index = index_xml_by_series(xml_paths)
print(f'{len(xml_index)} of them carry a SeriesInstanceUid and were indexed')
assert xml_index, ('parsed the XML but found no SeriesInstanceUid in any of '
                   'them -- inspect one with ET.parse(xml_paths[0]) before '
                   'going further')


In [ ]:
# 2-ii -- list LIDC-IDRI CT series from TCIA, then download a batch of those
# that have a matching annotation. No .tcia manifest needed: getSeries is the
# same query the Data Retriever's manifest was built from.
import shutil

from tqdm.auto import tqdm

NBIA = 'https://services.cancerimagingarchive.net/nbia-api/services/v1'
DICOM_ROOT = pathlib.Path('/kaggle/working/lidc_dicom')
DICOM_ROOT.mkdir(parents=True, exist_ok=True)

BATCH = 40   # CT series this pass, ~100-300 MB each. Start small, then raise.

r = requests.get(f'{NBIA}/getSeries',
                 params={'Collection': 'LIDC-IDRI', 'Modality': 'CT'}, timeout=120)
r.raise_for_status()
series = r.json()
print(f'{len(series)} CT series in LIDC-IDRI')

uids = [s['SeriesInstanceUID'] for s in series if 'SeriesInstanceUID' in s]
matched = [u for u in uids if u in xml_index]
print(f'{len(matched)} of them have a matching annotation XML')
assert matched, ('no overlap between the series list and the XML index -- '
                 'compare uids[:3] against list(xml_index)[:3]')

def download_series(uid, dest):
    dest.mkdir(parents=True, exist_ok=True)
    zpath = dest / 'series.zip'
    with requests.get(f'{NBIA}/getImage', params={'SeriesInstanceUID': uid},
                      stream=True, timeout=300) as resp:
        resp.raise_for_status()
        with open(zpath, 'wb') as out:
            shutil.copyfileobj(resp.raw, out)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(dest)
    zpath.unlink()

ok, failed = 0, []
for uid in tqdm(matched[:BATCH], desc='series'):
    d = DICOM_ROOT / uid
    if d.exists() and any(d.glob('*.dcm')):
        ok += 1                                  # already fetched on a rerun
        continue
    try:
        download_series(uid, d)
        shutil.copy(xml_index[uid], d / xml_index[uid].name)
        ok += 1
    except Exception as e:                                          # noqa: BLE001
        failed.append((uid, f'{type(e).__name__}: {e}'))

print(f'\n{ok}/{min(BATCH, len(matched))} series in {DICOM_ROOT}')
if failed:
    print(f'{len(failed)} failed, first: {failed[0]}')
assert ok > 0, 'nothing downloaded -- see the errors above'


In [ ]:
# 2-iii -- ingest the batch with pylidc, then free the raw DICOM.
DATA = '/kaggle/working/data/lidc'
WAREHOUSE = pathlib.Path('/kaggle/working/pylidc_warehouse')
WAREHOUSE.mkdir(parents=True, exist_ok=True)

pathlib.Path.home().joinpath('.pylidcrc').write_text(
    f'[dicom]\npath = {DICOM_ROOT}\nwarehouse_path = {WAREHOUSE}\n')

# Check "ingested N cases (M with nodule masks)" below: M should be close to N.
# M == 0 means pylidc is not pairing the XML with its series -- stop there
# rather than training on a dataset with no ground truth.
!python scripts/prepare_lidc.py --source pylidc \
    --limit {BATCH} --max-slice-thickness 1.5 \
    --out {DATA}

# The .npz cases are small; the raw DICOM is not. Drop it so the next batch
# fits. Comment out to keep it (e.g. to re-ingest at a different slice
# thickness without re-downloading).
shutil.rmtree(DICOM_ROOT, ignore_errors=True)
print(f'freed {DICOM_ROOT}')


## 3 · Train the prior

~48M params at 256², batch 8 with AMP ≈ 7 GB on a T4.

**Rerun this cell in a new session to resume** — it picks up from `last.pt`
with the optimiser, scaler, step and best-val intact. `--max-hours 11` stops
cleanly before Kaggle kills the session, so no progress is ever lost to the cap.

In [ ]:
!python scripts/train_prior.py \
    --data {DATA} \
    --out  /kaggle/working/runs/prior \
    --epochs 60 --batch-size 8 --base-dim 64 \
    --amp --workers 2 --max-hours 11

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv('/kaggle/working/runs/prior/log.csv')
ax = log.plot(x='epoch', y=['train_loss', 'val_loss'], figsize=(6, 3.4), grid=True)
ax.set_ylabel('eps-prediction MSE'); plt.tight_layout(); plt.show()
log.tail()

## 4 · Reconstruct

Baselines first — a learned prior has to beat them to justify its existence.
`--n-posterior 8` gives the mean reconstruction *and* the per-voxel uncertainty
map; it costs ~270 MB at 256³ and is the only thing that lets a reader tell
measured structure from structure the prior invented.

In [ ]:
# DATA was set in section 2 (LIDC-IDRI), or by the 1b smoke test
RECON = '/kaggle/working/runs/recon'
PRIOR = '/kaggle/working/runs/prior/best.pt'

for method in ['fbp', 'sirt_tv', 'cgls']:
    !python scripts/reconstruct.py --data {DATA} --out {RECON} \
        --track A16 --method {method} --n-iter 60 --device cuda

!python scripts/reconstruct.py --data {DATA} --out {RECON} \
    --track A16 --method diffusion --prior {PRIOR} \
    --n-steps 50 --n-posterior 8 --slice-batch 16 --device cuda

## 5 · Evaluate

Paired metrics, because this is a paired per-patient reconstruction task with
ground truth for every case — not FID/MMD, which measure distributional
similarity for *unconditional* generation. Nodule metrics are computed inside a
dilated bounding box, and PSNR/SSIM inside the lung mask.

In [ ]:
!python scripts/evaluate.py --recon {RECON} --data {DATA}

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'{RECON}/*/*.png'))[:6]:
    print(f.split('/')[-2], '·', f.split('/')[-1]); display(Image(f))

## 6 · The experiments that make the contribution

The architecture alone is not the novelty — R²-Gaussian, X-Gaussian, DOLCE and
DiffusionMBIR already occupy that ground. These three are:

1. **MDVC** — minimum detectable volume change vs view count and arc. Answers the
   Volume Doubling Time objective directly, and says where absolute volumetry
   stops being trustworthy.
2. **Hallucination audit** — insert / erase / present. Yields detection
   sensitivity and the **false-positive nodule rate**, which is the number a
   clinician cares about given the 96% LDCT false-positive rate.
3. **O5 ablation** — does the Gaussian parameterisation earn its place? A negative
   answer is a legitimate result.

> All three need the four-reader nodule masks, which is precisely why
> LIDC-IDRI is the dataset and not a centroid-only collection.


In [ ]:
!python scripts/mdvc.py --data {DATA} --out /kaggle/working/runs/mdvc \
    --tracks A32 A16 B15 --methods sirt_tv diffusion --prior {PRIOR} \
    --gains 0.05 0.10 0.20 0.30 0.50 --limit 5 --device cuda

In [ ]:
display(Image('/kaggle/working/runs/mdvc/mdvc.png'))

In [ ]:
!python scripts/hallucination.py --data {DATA} \
    --out /kaggle/working/runs/hallucination \
    --tracks A16 B15 --methods sirt_tv diffusion --prior {PRIOR} \
    --limit 5 --device cuda

In [ ]:
!python scripts/ablate_roi.py --data {DATA} \
    --out /kaggle/working/runs/ablate_roi \
    --track A16 --limit 3 --iters 800 --device cuda

## 7 · Collect outputs

Kaggle persists `/kaggle/working` (~20 GB). Checkpoints are large, so keep the
one you need and drop the rest before committing the notebook.

In [ ]:
import shutil, pathlib

for p in pathlib.Path('/kaggle/working/runs/prior').glob('last.pt'):
    print('consider deleting to save space:', p, f'{p.stat().st_size/2**20:.0f} MiB')

shutil.make_archive('/kaggle/working/medlift3d_results', 'zip',
                    '/kaggle/working/runs', )
print('bundled -> /kaggle/working/medlift3d_results.zip')